In [108]:
import pandas as pd 
import numpy as np 
df=pd.read_csv("cpm_predictor/data/data_input.csv")
df_geo=pd.read_csv("cpm_predictor/data/geotargets-2025-10-29.csv")

In [109]:
df["Geography Targeting - Include"]

0                                                    NaN
1                                                    NaN
2                                                    NaN
3                                                    NaN
4                                                    NaN
                             ...                        
140    20456; 1007764; 1007765; 1007820; 1007826; 905...
141    20456; 1007751; 1007764; 1007765; 1007820; 100...
142    1007785; 9040240; 9040241; 9040242; 9040243; 9...
143                                             1007768;
144                                                  NaN
Name: Geography Targeting - Include, Length: 145, dtype: object

In [110]:
# Build mapping
geo_map = (
    df_geo
    .set_index("Criteria ID")["Name"]
    .to_dict()
)

def map_geo_ids(cell):
    if pd.isna(cell):
        return cell

    ids = (
        str(cell)
        .split(";")
    )

    names = [
        geo_map.get(int(i.strip()), i.strip())
        for i in ids
        if i.strip().isdigit()
    ]

    return ", ".join(names) if names else None

df["Geography Targeting - Include"] = df["Geography Targeting - Include"].apply(map_geo_ids)


In [111]:
df["Geography Targeting - Include"] = (
    df["Geography Targeting - Include"]
    .replace(r"^\s*$", pd.NA, regex=True)
    .fillna(df["Markets"])
)

In [112]:
df["Geography Targeting - Include"].isna().sum()

np.int64(1)

In [113]:
print(df["TrueView Video Ad Formats"].value_counts(dropna=False))
print(df["Inventory Mode"].value_counts(dropna=False))
print(df["Video Ad Format"].value_counts(dropna=False))

TrueView Video Ad Formats
Skippable / Bumper / Non-skippable    103
NaN                                    30
Non Skippable                           9
Responsive                              3
Name: count, dtype: int64
Inventory Mode
Limited     90
NaN         30
Expanded    19
Standard     6
Name: count, dtype: int64
Video Ad Format
Non skippable      111
NaN                 30
Responsive           3
Efficient reach      1
Name: count, dtype: int64


In [114]:
def normalize_text(x):
    if not isinstance(x, str):
        return x
    return x.strip().replace("-", " ").title()

for col in [
    "TrueView Video Ad Formats",
    "Inventory Mode",
    "Video Ad Format"
]:
    df[col] = df[col].apply(normalize_text)

In [115]:
VALID_TRUEVIEW_FORMATS = {
    "Skippable / Bumper / Non Skippable",
    "Skippable",
    "Non Skippable",
    "Bumper"
}

VALID_INVENTORY_MODES = {
    "Limited",
    "Standard",
    "Expanded"
}

VALID_VIDEO_AD_FORMATS = {
    "Skippable",
    "Non Skippable",
    "Bumper"
}


In [116]:
import numpy as np

def realign_video_inventory_columns(row):
    tv = row["TrueView Video Ad Formats"]
    inv = row["Inventory Mode"]
    vid = row["Video Ad Format"]

    candidates = [
        v for v in [tv, inv, vid]
        if isinstance(v, str) and not v.isdigit()
    ]

    row["TrueView Video Ad Formats"] = next(
        (v for v in candidates if v in VALID_TRUEVIEW_FORMATS),
        np.nan
    )

    row["Inventory Mode"] = next(
        (v for v in candidates if v in VALID_INVENTORY_MODES),
        np.nan
    )

    row["Video Ad Format"] = next(
        (v for v in candidates if v in VALID_VIDEO_AD_FORMATS),
        np.nan
    )

    return row


In [117]:
df = df.apply(realign_video_inventory_columns, axis=1)

In [118]:
print("TrueView Video Ad Formats")
print(df["TrueView Video Ad Formats"].value_counts(dropna=False))

print("\nInventory Mode")
print(df["Inventory Mode"].value_counts(dropna=False))

print("\nVideo Ad Format")
print(df["Video Ad Format"].value_counts(dropna=False))


TrueView Video Ad Formats
TrueView Video Ad Formats
Skippable / Bumper / Non Skippable    103
NaN                                    33
Non Skippable                           9
Name: count, dtype: int64

Inventory Mode
Inventory Mode
Limited     90
NaN         30
Expanded    19
Standard     6
Name: count, dtype: int64

Video Ad Format
Video Ad Format
Non Skippable    111
NaN               34
Name: count, dtype: int64


In [106]:
nan_counts = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
)

print(nan_counts)

Video Ad Format                      34
TrueView Video Ad Formats            33
Subtype                              30
Type                                 30
Pacing Amount                        30
Pacing Rate                          30
Frequency Exposures                  30
Frequency Enabled                    30
Pacing                               30
Budget Type                          30
Frequency Period                     30
TrueView View Frequency Enabled      30
Partner Revenue Amount               30
Partner Revenue Model                30
TrueView View Frequency Exposures    30
TrueView View Frequency Period       30
Inventory Mode                       30
Device                                9
Start Date_1                          8
End Date_1                            8
Planned Freq                          6
Planned Reach 1+                      6
Planned Budget                        6
Planned Impressions                   6
TG                                    6


In [119]:
import pandas as pd
import numpy as np
import re
from dateutil import parser

def parse_mixed_date(x):
    if pd.isna(x):
        return pd.NaT

    # Excel serial number
    if isinstance(x, (int, float)) and x > 30000:
        try:
            return pd.to_datetime("1899-12-30") + pd.to_timedelta(int(x), unit="D")
        except Exception:
            return pd.NaT

    # String date
    if isinstance(x, str):
        x = x.strip()
        if x == "":
            return pd.NaT

        # remove ordinal suffixes
        x = re.sub(r"(\d+)(st|nd|rd|th)", r"\1", x)

        try:
            return parser.parse(x, dayfirst=True, fuzzy=True)
        except Exception:
            return pd.NaT

    return pd.NaT


# 🔥 FORCE conversion (overwrite column)
df["Start Date_1"] = df["Start Date_1"].apply(parse_mixed_date)
df["End Date_1"] = df["End Date_1"].apply(parse_mixed_date)

# 🔍 VERIFY TYPES (THIS MUST SHOW datetime64)
print(df["Start Date_1"].dtype)
print(df["End Date_1"].dtype)


datetime64[ns]
datetime64[ns]


In [120]:
def build_month_range(start, end):
    if not isinstance(start, pd.Timestamp) or not isinstance(end, pd.Timestamp):
        return pd.NA

    sm, em = start.strftime("%b"), end.strftime("%b")
    sy, ey = start.year, end.year

    if sm == em and sy == ey:
        return f"{sm} {sy}"
    if sy == ey:
        return f"{sm}–{em} {sy}"
    return f"{sm} {sy}–{em} {ey}"


In [121]:
df["month_range"] = df.apply(
    lambda r: build_month_range(r["Start Date_1"], r["End Date_1"]),
    axis=1
)

print("month_range NaN:", df["month_range"].isna().sum())


month_range NaN: 8


In [122]:

# Create campaign duration (in days)
df["campaign_duration_days"] = (df["End Date_1"] - df["Start Date_1"]).dt.days
df = df.drop(columns=["Start Date_1", "End Date_1"])


# -----------------------------
# 2. Drop Markets column
# -----------------------------
df = df.drop(columns=["Markets"], errors="ignore")

# -----------------------------
# 3. Split numerical and non-numerical columns
# -----------------------------
numerical_cols = df.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = df.select_dtypes(exclude=["number"]).columns.tolist()

print("Numerical columns:")
print(numerical_cols)

print("\nNon-numerical (categorical) columns:")
print(categorical_cols)

# -----------------------------
# 4. Print number of unique values for categorical columns
# -----------------------------
unique_counts = (
    df[categorical_cols]
    .nunique(dropna=True)
    .sort_values(ascending=False)
)

print("\nUnique value count (categorical columns):")
print(unique_counts)


Numerical columns:
['Planned Freq', 'Planned Budget', 'Pacing Amount', 'Frequency Exposures', 'TrueView View Frequency Exposures', 'Partner Revenue Amount', 'campaign_duration_days']

Non-numerical (categorical) columns:
['Device', 'TG', 'Planned Reach 1+', 'Planned Impressions', 'Del Cpm/\nBidvid  cpm', 'Type', 'Subtype', 'Budget Type', 'Pacing', 'Pacing Rate', 'Frequency Enabled', 'Frequency Period', 'TrueView View Frequency Enabled', 'TrueView View Frequency Period', 'Partner Revenue Model', 'Geography Targeting - Include', 'TrueView Video Ad Formats', 'Inventory Mode', 'Video Ad Format', 'month_range']

Unique value count (categorical columns):
Planned Impressions                136
Planned Reach 1+                   132
Geography Targeting - Include       78
Del Cpm/\nBidvid  cpm               78
TG                                  20
Device                              14
month_range                          8
Subtype                              3
Inventory Mode                 

In [123]:
print(df["Planned Impressions"].dtype)
print(df["Planned Reach 1+"].dtype)
print(df["Del Cpm/\nBidvid  cpm"].dtype)

object
object
object


In [124]:
NUMERIC_COLUMNS = [
    "Planned Impressions",
    "Planned Reach 1+",
    "Planned Budget",
    "Planned Freq",
    "Pacing Amount",
    "Pacing Rate",
    "Partner Revenue Amount",
    "Del Cpm/\nBidvid  cpm"
]
for col in NUMERIC_COLUMNS:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.replace("#REF!", "", regex=False)
            .str.replace("NA", "", regex=False)
            .str.strip()
            .pipe(pd.to_numeric, errors="coerce")
        )

In [125]:
numerical_cols = df.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = df.select_dtypes(exclude=["number"]).columns.tolist()

print("Numerical columns:")
print(numerical_cols)

print("\nCategorical columns:")
print(categorical_cols)


Numerical columns:
['Planned Reach 1+', 'Planned Freq', 'Planned Budget', 'Planned Impressions', 'Del Cpm/\nBidvid  cpm', 'Pacing Rate', 'Pacing Amount', 'Frequency Exposures', 'TrueView View Frequency Exposures', 'Partner Revenue Amount', 'campaign_duration_days']

Categorical columns:
['Device', 'TG', 'Type', 'Subtype', 'Budget Type', 'Pacing', 'Frequency Enabled', 'Frequency Period', 'TrueView View Frequency Enabled', 'TrueView View Frequency Period', 'Partner Revenue Model', 'Geography Targeting - Include', 'TrueView Video Ad Formats', 'Inventory Mode', 'Video Ad Format', 'month_range']


In [126]:

# -----------------------------
# 3. Split numerical and non-numerical columns
# -----------------------------
numerical_cols = df.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = df.select_dtypes(exclude=["number"]).columns.tolist()

print("Numerical columns:")
print(numerical_cols)

print("\nNon-numerical (categorical) columns:")
print(categorical_cols)

# -----------------------------
# 4. Print number of unique values for categorical columns
# -----------------------------
unique_counts = (
    df[categorical_cols]
    .nunique(dropna=True)
    .sort_values(ascending=False)
)

print("\nUnique value count (categorical columns):")
print(unique_counts)


Numerical columns:
['Planned Reach 1+', 'Planned Freq', 'Planned Budget', 'Planned Impressions', 'Del Cpm/\nBidvid  cpm', 'Pacing Rate', 'Pacing Amount', 'Frequency Exposures', 'TrueView View Frequency Exposures', 'Partner Revenue Amount', 'campaign_duration_days']

Non-numerical (categorical) columns:
['Device', 'TG', 'Type', 'Subtype', 'Budget Type', 'Pacing', 'Frequency Enabled', 'Frequency Period', 'TrueView View Frequency Enabled', 'TrueView View Frequency Period', 'Partner Revenue Model', 'Geography Targeting - Include', 'TrueView Video Ad Formats', 'Inventory Mode', 'Video Ad Format', 'month_range']

Unique value count (categorical columns):
Geography Targeting - Include      78
TG                                 20
Device                             14
month_range                         8
Subtype                             3
Inventory Mode                      3
Frequency Enabled                   2
Frequency Period                    2
TrueView Video Ad Formats           2
B

In [129]:
df.columns

Index(['Device', 'TG', 'Planned Reach 1+', 'Planned Freq', 'Planned Budget',
       'Planned Impressions', 'Del Cpm/\nBidvid  cpm', 'Type', 'Subtype',
       'Budget Type', 'Pacing', 'Pacing Rate', 'Pacing Amount',
       'Frequency Enabled', 'Frequency Exposures', 'Frequency Period',
       'TrueView View Frequency Enabled', 'TrueView View Frequency Exposures',
       'TrueView View Frequency Period', 'Partner Revenue Model',
       'Partner Revenue Amount', 'Geography Targeting - Include',
       'TrueView Video Ad Formats', 'Inventory Mode', 'Video Ad Format',
       'month_range', 'campaign_duration_days'],
      dtype='object')

In [130]:
NUMERIC_COLUMNS = [
    "Planned Reach 1+",
    "Planned Freq",
    "Planned Budget",
    "Planned Impressions",
    "Pacing Rate",
    "Pacing Amount",
    "Partner Revenue Amount",
    "campaign_duration_days",
    "Del Cpm/\nBidvid  cpm"
]

df[NUMERIC_COLUMNS].dtypes


Planned Reach 1+          float64
Planned Freq              float64
Planned Budget            float64
Planned Impressions       float64
Pacing Rate               float64
Pacing Amount             float64
Partner Revenue Amount    float64
campaign_duration_days    float64
Del Cpm/\nBidvid  cpm     float64
dtype: object

In [134]:
bad_impr = df[df["Planned Impressions"] < 0]

print("Rows with negative planned impressions:", len(bad_impr))
bad_impr[[
    "Planned Impressions",
    "Planned Reach 1+",
    "Planned Budget",
    "Del Cpm/\nBidvid  cpm"
]].head()


Rows with negative planned impressions: 0


,Planned Impressions,Planned Reach 1+,Planned Budget,Del Cpm/\nBidvid cpm


In [136]:
df.to_csv("cpm_predictor/data/data_input.csv", index=False)